# Player Neural Network
We are going on an approach to deduce a "worth" or overall quantitative value to a player during a regular season in order to later make a prediction based on.

## Utils

In [10]:
def convert_int_season_to_str(season):
    if isinstance(season, int):
        return f"{season}-{season%2000 +1 :02d}" 
    return season

# Imports 

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import itertools 
from typing import Dict,Tuple
from IPython.display import display
import sklearn
from xgboost import XGBRegressor
from utils import getMatchAndPlayerStats,getMatchupByTeamBySeason,calculate_elo_rating,aggregate_matchup_data

from dotenv import load_dotenv
import os
load_dotenv()
DATA_FOLDER = os.getenv("DATA_FOLDER", "./datasets/DATA_AGGREGATIONS/")
MODEL_FOLDER = os.getenv("MODEL_FOLDER", "./models/")
pd.set_option('future.no_silent_downcasting', True)

In [12]:
NUM_GAMES=82
teams=['DAL','MIL','ATL','DEN','HOU','IND','OKC','CHI','ORL','BOS','DET','NYK'
,'CHA','LAL','SAC','MIA','LAC','GSW','POR','MIN','WAS','BKN','MEM','SAS'
,'PHX','NOP','UTA','TOR','PHI','CLE']
all_possible_matchups=list(itertools.combinations(teams, 2))
regular_games_total=pd.read_csv("./datasets/NBA_DATA_2010_2024/regular_season_totals_2010_2024.csv",delimiter=',',header=0)
regular_season_all_parts=pd.concat([
        pd.read_csv("./datasets/NBA_DATA_2010_2024/regular_season_box_scores_2010_2024_part_1.csv",delimiter=',',header=0),
        pd.read_csv("./datasets/NBA_DATA_2010_2024/regular_season_box_scores_2010_2024_part_2.csv",delimiter=',',header=0),
        pd.read_csv("./datasets/NBA_DATA_2010_2024/regular_season_box_scores_2010_2024_part_3.csv",delimiter=',',header=0)])

# NBA facts
Regular season each team makes 82 games.
The Best 8 teams of each conference (WEST & EAST), makes to the playoffs.
The goal with this model is to predict the probability of the winning a game between a specific matchup.
## Important Game Features 
- Home/ Away Game 
- Players List 
## Important Player Features
- season (season_year)
- time played (MIN)
- Field Goal Made (fieldGoalsMade)
- Field Goal Percentage (fieldGoalsPercentage)
- Field Goal Made 3PT (treePointersMade)
- Field Goal Percentage 3PT (threePointersPercentage)
- Free throw made (freeThrowsMade)
- Free Throw (percentagefreeThrowsPercentage)
- assists
- rebounds 
- steals 
- turnovers
- foulsPersonal
- blocks 
- points 
- plusMinusPoints

# Main Model Data Aggregation 
## Aggregate and save players data

In [15]:
all_player_stats=pd.DataFrame()
for season in range(2010,2024):
    playersStats=getMatchAndPlayerStats(regular_games_total, regular_season_all_parts,season=season,filterFields=["personName","season_year","teamTricode","WL","minutesParsed","points","fieldGoalsPercentage","threePointersPercentage","reboundsTotal","foulsPersonal","turnovers","fieldGoalsMade","fieldGoalsAttempted","steals","gamesPlayed"])
    scaler = sklearn.preprocessing.StandardScaler()
    playersStats_scaled = scaler.fit_transform(playersStats.drop(columns=['personName', 'teamTricode','season_year']))
    model1 = XGBRegressor()
    model1.load_model(MODEL_FOLDER+"xgb_tunned.json")
    y_pred_loaded = model1.predict(playersStats_scaled)
    # append the impact stat to the players stats
    playersStats["playerImpact"]=y_pred_loaded
    all_player_stats= pd.concat([all_player_stats,playersStats])
all_player_stats.to_csv(DATA_FOLDER+'playerStats.csv', index=False)

## Aggregate games and elo rating of each team

In [16]:
mt=pd.DataFrame([])
all_elos = pd.DataFrame([])
df_com_elo = calculate_elo_rating(regular_games_total)
df_com_elo = df_com_elo.sort_values(by=['SEASON_YEAR','TEAM_ABBREVIATION', 'GAME_DATE'])
df_com_elo['elo_last_10_avg'] = (
    df_com_elo
    .groupby('TEAM_ABBREVIATION')['elo_before_game']
    .transform(lambda x: x.shift(1).rolling(window=10, min_periods=1).mean())
)
for season in range(2010,2024):
    for team in teams:
            single_team=df_com_elo[df_com_elo['TEAM_ABBREVIATION']==str(team)][['GAME_ID','SEASON_YEAR','TEAM_ABBREVIATION','GAME_DATE', 'elo_before_game', 'elo_last_10_avg']]
            all_elos=pd.concat([all_elos,single_team[single_team['SEASON_YEAR']==convert_int_season_to_str(season)].tail(1)])
all_elos.to_csv(DATA_FOLDER+'gamesAndEloStats.csv',index=False)